CELL 1: KHỞI TẠO MÔI TRƯỜNG (Cài đặt & Dọn dẹp cache)

In [1]:
# Gỡ sạch các gói phân mảnh cũ để tránh xung đột thư viện
!rm -rf /kaggle/working/unsloth_compiled_cache
!rm -rf outputs
!pip uninstall -y unsloth unsloth-zoo trl transformers accelerate bitsandbytes xformers peft

# Cài đặt các phiên bản stable lõi từ PyPI
!pip install --no-cache-dir transformers==4.46.3 trl==0.12.0 accelerate==1.0.1 bitsandbytes peft datasets matplotlib tqdm

# Cài đặt Unsloth phân phối chính thức
!pip install --no-cache-dir unsloth

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 331.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 367.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 292.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 383.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 224.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 337.5 MB/s eta 0:00:00
  

CELL 2: TIỀN XỬ LÝ & CHUẨN HÓA DATASET Y KHOA

In [2]:
import os
import gc
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 1024
dtype = None
load_in_4bit = True

# 1. Khởi tạo Tokenizer & Base Model
print("🤖 Đang nạp Base Model Qwen-2.5-7B-Instruct...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = {"": 0}
)

# Cấu hình Chat Template chuẩn dòng Qwen-2.5
tokenizer = get_chat_template(tokenizer, chat_template = "qwen-2.5")
tokenizer.pad_token       = tokenizer.eos_token
tokenizer.pad_token_id    = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.eos_token_id

# 2. Định nghĩa hàm tiền xử lý đóng gói chuẩn Y khoa (Loại bỏ từ ngữ rác)
def formatting_prompts_func(examples):
    convos = examples["messages"]
    cleaned_texts = []
    
    for convo in convos:
        # Ép cấu trúc một System Prompt nghiêm túc để định hình tri thức bác sĩ chuyên khoa
        system_prompt = {
            "role": "system",
            "content": "Bạn là một Bác sĩ AI chuyên khoa Hô hấp. Hãy trả lời câu hỏi của người bệnh bằng tiếng Việt chuẩn y khoa, ngắn gọn, chính xác, không lặp từ và không sử dụng thuật ngữ dịch máy thô sơ."
        }
        
        # Lọc bỏ các tin nhắn hệ thống cũ nếu có trong file json và chèn prompt chuẩn vào đầu
        new_convo = [system_prompt]
        for msg in convo:
            if msg["role"] != "system":
                new_convo.append(msg)
                
        # Áp dụng Chat Template
        formatted_text = tokenizer.apply_chat_template(new_convo, tokenize=False, add_generation_prompt=False)
        cleaned_texts.append(formatted_text)
        
    return { "text" : cleaned_texts }

# 3. Tiến hành Tokenize dữ liệu chuyển sang Tensor
print("📦 Đang tiến hành tiền xử lý và số hóa Dataset...")
dataset_train = load_dataset("json", data_files="pneumonia_train_chat.json", split="train")

def tokenize_function(examples):
    processed = formatting_prompts_func(examples)
    tokenized = tokenizer(processed["text"], truncation=True, max_length=max_seq_length, padding=False)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

dataset_tokenized = dataset_train.map(tokenize_function, batched=True, remove_columns=dataset_train.column_names)
dataset_tokenized.set_format("torch")
print(f"✅ Đã chuẩn bị xong {len(dataset_tokenized)} mẫu dữ liệu huấn luyện đạt chuẩn y khoa.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🤖 Đang nạp Base Model Qwen-2.5-7B-Instruct...
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
📦 Đang tiến hành tiền xử lý và số hóa Dataset...


FileNotFoundError: Unable to find '/kaggle/working/pneumonia_train_chat.json'

Cell 3: Cấu hình LoRA + Cấu hình Accelerator an toàn + Vòng lặp huấn luyện 100% Steps PyTorch và vẽ đồ thị Loss Curve.

In [ ]:
from transformers import DataCollatorForSeq2Seq, TrainingArguments
from torch.utils.data import DataLoader
from accelerate import Accelerator
import bitsandbytes as bnb
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ===========================================================================
# 1. CẤU HÌNH THAM SỐ LỚP TOÁN TỬ LORA (PEFT)
# ===========================================================================
print("⚙️ Đang cấu hình các tham số lớp toán tử LoRA (PEFT)...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", # Giữ nguyên Unsloth để tiết kiệm VRAM
    random_state = 3407,
    use_rslora = False,  
)

model.train() # Chuyển mô hình sang trạng thái học tập

# Khởi tạo bộ gom dữ liệu động (Data Collator) và DataLoader
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)
train_dataloader = DataLoader(dataset_tokenized, shuffle=True, batch_size=1, collate_fn=data_collator)

# ===========================================================================
# 2. KHỞI TẠO TRAINING ARGUMENTS ĐỂ QUẢN LÝ GRADIENT ACCUMULATION AN TOÀN
# ===========================================================================
max_steps = 220
gradient_accumulation_steps = 8

# Sử dụng TrainingArguments gốc để Hugging Face tự quản lý luồng dồn tích lũy gradient
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=2e-4,
    logging_steps=5,
    max_steps=max_steps,
    output_dir="outputs",
    report_to="none"
)

# Khởi tạo Accelerator đồng bộ theo tham số cấu hình sạch
accelerator = Accelerator(
    mixed_precision="fp16" if not torch.cuda.is_bf16_supported() else "bf16",
    gradient_accumulation_steps=gradient_accumulation_steps # Ép quản lý dồn tích lũy ở tầng lõi
)

optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=training_args.learning_rate, weight_decay=0.01)

# Đồng bộ phần cứng thông qua nền tảng Accelerator
model, optimizer, train_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader
)

# ===========================================================================
# 3. VÒNG LẶP HUẤN LUYỆN CHUẨN AN TOÀN (DIỆT TẬN GỐC LỖI INPLACE)
# ===========================================================================
print("🔥 HỆ THỐNG BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN CHUYÊN SÂU VIÊM PHỔI (100% STEPS)...")

steps = []
losses = []
actual_step = 0
running_loss = 0.0

progress_bar = tqdm(total=max_steps, desc="Training Progress")

# Sử dụng biến kiểm soát vòng lặp an toàn
completed = False
while actual_step < max_steps and not completed:
    for batch in train_dataloader:
        # Sử dụng ngữ cảnh accumulate của accelerator để tránh can thiệp thủ công vào loss tensor
        with accelerator.accumulate(model):
            outputs = model(**batch)
            loss = outputs.loss
            
            running_loss += loss.item()
            
            # Tính toán backward trực tiếp trên loss nguyên bản (Tránh lỗi toán tử đồ thị)
            accelerator.backward(loss)
            
            # Cập nhật trọng số khi accelerator gom đủ accumulation steps
            if accelerator.sync_gradients:
                optimizer.step()
                optimizer.zero_grad()
                
                actual_step += 1
                
                # Ghi nhận log loss sau mỗi 5 step thực tế để vẽ đồ thị
                if actual_step % 5 == 0:
                    steps.append(actual_step)
                    losses.append(running_loss / gradient_accumulation_steps)
                    progress_bar.set_postfix({"Loss": f"{losses[-1]:.4f}"})
                
                running_loss = 0.0
                progress_bar.update(1)
                
                if actual_step >= max_steps:
                    completed = True
                    break

progress_bar.close()
print("🎉 Quá trình huấn luyện mạng Neural hoàn tất thành công 100% chặng đường!")

# ===========================================================================
# 4. VẼ ĐỒ THỊ LOSS CURVE CHUẨN XÁC CHO ĐATN
# ===========================================================================
plt.figure(figsize=(10, 5))
plt.plot(steps, losses, marker='o', color='#2ca02c', linewidth=2, markersize=5, label='Training Loss')
plt.title('Training Loss Convergence Curve (Đồ thị hội tụ Loss toàn diện)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Huấn luyện qua các bước thực tế (Steps)', fontsize=12)
plt.ylabel('Giá trị Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

CELL 4: ĐÁNH GIÁ THỰC NGHIỆM VÀ SUY LUẬN AN TOÀN (Inference)

In [ ]:
print("\n🩺 Chuyển đổi trạng thái mô hình sang chế độ suy luận chuyên sâu...")
unwrapped_model = accelerator.unwrap_model(model)
FastLanguageModel.for_inference(unwrapped_model)

# Đặt câu hỏi thực nghiệm lâm sàng phức tạp để kiểm tra kỹ năng của AI
cau_hoi_kiem_tra = "Bệnh nhân bị viêm phổi cấp tính thường có những biểu hiện lâm sàng đặc trưng nào?"

# Tạo cấu trúc hội thoại đúng với định dạng đầu vào đã huấn luyện
messages = [
    {
        "role": "system",
        "content": "Bạn là một Bác sĩ AI chuyên khoa Hô hấp. Hãy trả lời câu hỏi của người bệnh bằng tiếng Việt chuẩn y khoa, ngắn gọn, chính xác, không lặp từ và không sử dụng thuật ngữ dịch máy thô sơ."
    },
    {"role": "user", "content": cau_hoi_kiem_tra}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

print("🩺 Bác sĩ AI đang xử lý dữ liệu và đưa ra kết quả phân tích...")
print("\n" + "="*70 + "\n[KẾT QUẢ SUY LUẬN TRỰC QUAN ĐỂ ĐƯA VÀO ĐATN]\n" + "="*70)

with torch.no_grad():
    outputs = unwrapped_model.generate(
        input_ids=inputs,
        max_new_tokens=350,       # Token giới hạn vừa đủ để cô đọng cấu trúc câu trả lời
        temperature=0.2,          # Đặt cực thấp để triệt tiêu hoàn toàn sự bịa đặt kiến thức
        top_p=0.85,
        repetition_penalty=1.2,   # Phạt nặng để ngăn chặn mô hình lặp lại từ hoặc sinh ký tự rác
        eos_token_id=tokenizer.eos_token_id
    )

# Giải mã kết quả đầu ra sạch
ket_qua_sach = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Lọc bỏ đoạn prompt hệ thống hiển thị thô để lấy phần văn bản bác sĩ trả lời
print(ket_qua_sach.split("assistant")[-1].strip())
print("\n" + "="*70)


# ===========================================================================
# 💾 ĐOẠN ĐÃ SỬA: LƯU ADAPTER LORA SIÊU NHẸ ĐỂ CHỐNG TRÀN Ổ ĐĨA KAGGLE
# ===========================================================================
print("💾 Đang tiến hành đóng gói và lưu riêng cấu hình trọng số LoRA...")
try:
    # Chỉ lưu các ma trận thích ứng LoRA tinh chỉnh và cấu hình Tokenizer đi kèm
    unwrapped_model.save_pretrained("medical_ai_lora_adapter")
    tokenizer.save_pretrained("medical_ai_lora_adapter")
    print("🎉 Thành công! Cấu hình trọng số LoRA đã được lưu tại thư mục 'medical_ai_lora_adapter'.")
    print("💡 Mẹo ĐATN: File lưu trữ cực nhẹ (vài chục MB), không làm tràn ổ đĩa Kaggle, sẵn sàng nộp hội đồng!")
except Exception as e:
    print(f"❌ Có lỗi phát sinh khi lưu: {e}")
# ===========================================================================

In [ ]:
# Nén toàn bộ thư mục thành file medical_ai_adapter.zip
!zip -r medical_ai_adapter.zip /kaggle/working/medical_ai_lora_adapter